## 🔰PyTorchでニューラルネットワーク基礎　#39 【GPT編・分類問題】


### 内容
* Qiitaの記事と連動しています
* 各種ファイルの保存先は環境によって適宜変更してください


### データについて
* Livedoorニュースコーパスのlivedoor-hommeカテゴリを利用します。
* huggingfaceの　llm-book/livedoor-news-corpus　などから適宜ダウンロードしてください。
* 先頭から3行目のタイトル部分のテキストを利用します。

### トークナイザーについて
* tokenizer/livedoor_homme_tokenizer_8k.json
    * bytelevel BPEで構成した語彙数8kのtokenizer


### 今回扱う内容
1. GPTタイプのモデルによるテキスト分類
    * 分類したいテキストの文末に分類用のトークン（今回は \<eod\>で代用）を追加
    * 分類用トークンを利用して通常のテキスト分類の形に持ち込みます
2. 日本語データでの学習

### 注意点
* 汎用的な内容でのテキスト分類については、事前学習の効果が発揮されません

In [1]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from sklearn.model_selection import train_test_split

# Dataset・DataLoaderカスタマイズ時に利用
from tokenizers.processors import TemplateProcessing  # トークナイザーのencodeテンプレート
from torch.nn.utils.rnn import pad_sequence           # paddingして等長化するときに使う
from functools import partial                     # collate_fnのときに使う


# 精度を計算する関数
def accuracy(y, t):
    _, argmax_list = torch.max(y, dim=1)
    accuracy = sum(argmax_list == t).item()/len(t)
    return accuracy


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"{device=}")

device=device(type='cuda')


In [2]:
# ファイル名
data_filename = "./data/title_class5.jsonl"   # 分類データ
tokenizer_filename = "tokenizer/livedoor_home_tokenizer_8k.json"
pretrain_filename = "model/seq_512_bpe_8k.model"
cls_model_filename = "model/homme_classification_5.model"

# トークナイザーの確認
tokenizer = Tokenizer.from_file(tokenizer_filename)

print(f"<pad>: {tokenizer.token_to_id('<pad>')}")
print(f"<eod>: {tokenizer.token_to_id('<eod>')}")

print(f"size: {tokenizer.get_vocab_size()}")

<pad>: 0
<eod>: 1
size: 8000


### データの確認
* ライセンスの関係からデータが添付されていません!!
* 今回利用するのは次の表のtitleとlabel_id列を利用します。
* hommeカテゴリーのタイトルを事前に5種類に分類しておきます。
* 記事ではHuggingFaceの **cl-nagoya/ruri-v3-130m/** を利用して、タイトル部分の埋め込みベクトルを作成 (もちろん他のモデルでもOK👍)cos類似度を利用してタイトルを適当に分類しました。

In [3]:
df_org = pd.read_json(data_filename, lines=True)
df, df_test = train_test_split(df_org, stratify=df_org["label_id"], random_state=55)
df.head(3)

,title,label,label_id
152,「会社が携帯代を出してくれない」-辛口説教部屋vol.57,人生相談,2
162,年収1000万円のビジネスマンに聞いた「あなたは市場価値をどれぐらい意識していますか?」-年...,キャリア,1
48,"キン肉マンTシャツ企画、一番人気は名脇役として知られる“あの超人""!!",ファッション,0


## GPTタイプモデル

In [4]:
class ModelConfig:
    def __init__(self, tokenizer):
        # モデル構造
        self.vocab_size = tokenizer.get_vocab_size()
        self.seq_len = 512   # 128トークンだとSFT時に少ない
        self.d_model = 256   # 512
        self.nhead = 8
        self.dim_feedforward = 4*self.d_model
        self.num_layers = 6
        self.dropout = 0.1
        
        # 特殊トークンID
        self.pad_token_id = tokenizer.token_to_id("<pad>")
        self.eod_token_id = tokenizer.token_to_id("<eod>")

        # 学習データに関する設定
        self.context_size = self.seq_len         # 学習できる長さ
        self.context_stride = self.context_size  # 重なり具合の調整

        # 学習設定
        self.batch_size = 128
        self.ignore_index = -100
        self.learning_rate = 0.001  # これだとデフォルトと変わらない
        self.num_epochs = 30
        self.max_grad_norm = 1.0

    # 属性を追加・更新するメソッド
    # 設定時のタイポに注意だぞ〜
    def update(self, **kwargs):
        """渡されたキーワード引数で設定を動的に追加・更新する"""
        for key, value in kwargs.items():
            setattr(self, key, value)

In [5]:
class DNN(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config
        
        self.token_embedding = nn.Embedding(num_embeddings=config.vocab_size, embedding_dim=config.d_model,  padding_idx=config.pad_token_id)
        self.pos_embedding = nn.Embedding(num_embeddings=config.seq_len, embedding_dim=config.d_model)
        self.dropout = nn.Dropout(config.dropout)

        transformer_layer = nn.TransformerEncoderLayer(
            d_model=config.d_model,
            nhead=config.nhead,
            dim_feedforward=config.dim_feedforward,
            dropout=config.dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.transformer = nn.TransformerEncoder(transformer_layer, num_layers=config.num_layers, enable_nested_tensor=False)

        self.layer_norm = nn.LayerNorm(config.d_model)
        self.cls_head = nn.Linear(config.d_model, config.num_labels)

    def forward(self, input_ids, attention_mask):
        batch_size, seq_len = input_ids.shape
        positions = torch.arange(seq_len, device=input_ids.device)

        tok_emb = self.token_embedding(input_ids)
        pos_emb = self.pos_embedding(positions).unsqueeze(0)
        x = tok_emb + pos_emb
        x = self.dropout(x)
        # nn.Transformer.generate_square_subsequent_mask を使ってマスクを生成
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len, dtype=torch.bool, device=x.device)

        key_padding_mask = (attention_mask == 0)

        # 自己回帰型 transformer (transformer decoder)
        x = self.transformer(x, mask=causal_mask, src_key_padding_mask=key_padding_mask, is_causal=True)
        x = self.layer_norm(x)

        sequence_lengths = attention_mask.sum(dim=1).long()
        eod_positions = sequence_lengths - 1
        batch_indices = torch.arange(batch_size, device=input_ids.device)
        eod_states = x[batch_indices, eod_positions, : ]   # [B, T, d_model] -> [B, d_model]
        logits = self.cls_head(eod_states)                 # [B, d_model] -> [B, num_labels]
        return logits

### 事前学習したモデルを読み込む部分
* torch.loadでモデルを読み込む
* config.updateで設定変更や追加
* model_sd以降の行で最終のLinearを除いた重みをコピー
    * おかしいぞ〜: [] となれば問題なし

In [ ]:
# チェックポイント読み込み
checkpoint = torch.load(pretrain_filename, map_location=device, weights_only=False)

config = ModelConfig(tokenizer)
config.__dict__.update(checkpoint["config"])   # 事前学習時の構造を復元

# configの追加設定と数値の変更
config.update(
    num_labels    = 5,      # 分類数
    batch_size    = 16,
    learning_rate = 5e-4,
    num_steps    = 150,    # 100ステップでいいかも
)


model = DNN(config).to(device)

# 形が一致するパラメータだけを拾う（fc.* は自動的に落ちる）
model_sd = model.state_dict()
pretrained_sd = {
    k: v for k, v in checkpoint["model_state_dict"].items()
    if k in model_sd and model_sd[k].shape == v.shape
}

# 必要な部分（事前学習モデルの一部分）の重みだけ必要なのでstrict=False にする。
missing, unexpected = model.load_state_dict(pretrained_sd, strict=False)
print("変更部分:", missing)      # ['cls_head.weight', 'cls_head.bias'] だけのはず
print("おかしいぞ〜:", unexpected)   # [] のはず

変更部分: ['cls_head.weight', 'cls_head.bias']
おかしいぞ〜: []


* 最後の分類アダプターである全結合層のみ学習させてみる。
* 事前学習したパラメータは一切変更しない！

In [7]:
params_to_update = []

for name , param in model.named_parameters():
    param.requires_grad = False
#for name, param in model.transformer_encoder.layers[-2:].named_parameters():  # 最終層＋１
#    param.requires_grad = True
#    params_to_update.append(param)
#    print("更新されるパラメータ:", name)
for name, param in model.cls_head.named_parameters():
    param.requires_grad = True
    params_to_update.append(param)
    print("更新されるパラメータ:", name) 

更新されるパラメータ: weight
更新されるパラメータ: bias


In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(params_to_update,lr=config.learning_rate)

### 学習データの読み込み準備

In [ ]:
# (1) 深い意味がないけど置き換えてみた　直接記入したほうがいいかも
eod_id = tokenizer.token_to_id("<eod>")

# (2) encodeするときのテンプレ
tokenizer.post_processor = TemplateProcessing(
    single="$A <eod>",
    special_tokens=[("<eod>", eod_id)],
)


class ClassificationDataset(Dataset):
    def __init__(self, data, tokenizer):
        # (3) encode_batchで一度にid化
        encodings = tokenizer.encode_batch(list(data["title"]))
        ids_list = [torch.tensor(e.ids, dtype=torch.long) for e in encodings]

        # (5) ラベル
        labels = torch.tensor(data["label_id"].tolist(), dtype=torch.long)        
        
        self.labels = labels
        self.ids_list = ids_list
   
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, index):
        return {"ids": self.ids_list[index],
                "label": self.labels[index]
                }


def padding_collate_fn(batch, tokenizer=tokenizer):
    
    # (1) tokenizerを使い<pad> IDを求める
    padding_id = tokenizer.token_to_id("<pad>")
    # (2) 入力されるデータがtensorなのでそのまま活用
    ids_list = [x["ids"] for x in batch]
    labels = torch.stack([x["label"] for x in batch])
    # (3) padding
    padded_ids = pad_sequence(ids_list, batch_first=True, padding_value=padding_id)
    # (4) padマスクの作成
    attention_mask = (padded_ids != padding_id).long()  # 実トークン=1, <pad>=0
    # (5)
    return {"ids": padded_ids, "attention_mask": attention_mask, "label": labels}


collate_wrapper = partial(padding_collate_fn, tokenizer=tokenizer)

In [ ]:
# (1) DatasetとDataLoader
dataset = ClassificationDataset(data=df, tokenizer=tokenizer)
dataloader = DataLoader(
    dataset = dataset, 
    batch_size=config.batch_size,
    shuffle = True,
    collate_fn=collate_wrapper,
    pin_memory=torch.cuda.is_available(), # GPU使う時True
    drop_last = True
    )

# (2) step数で変数更新
def infinite_loader(dataloader):
    while True:
        for batch in dataloader:
            yield batch

data_iter = infinite_loader(dataloader)  # step数で計測

model.train()
max_iter = config.num_steps

# GPUが対応している場合は、bf16へ変更してflash attentionを使う
# bf16を利用し、SDPAが利用可能な最適ものを選択
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

# (3) 更新ループ
for step in range(max_iter):
    batch = next(data_iter)
    input_ids = batch["ids"].to(device, non_blocking=True)
    attention_mask = batch["attention_mask"].to(device, non_blocking=True)
    label    = batch["label"].to(device, non_blocking=True)
    optimizer.zero_grad()
    with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=use_bf16):
        y = model(input_ids, attention_mask)
        loss = criterion(y, label)
        acc = accuracy(y, label)

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)
    optimizer.step()
    if (step+1)%50 == 0:
        print(f"step: {step+1}/{max_iter} | Loss: {loss.item():.4f} | acc: {acc:.4f} ")
#
# step: 50/150 | Loss: 1.1976 | acc: 0.5625 
# step: 100/150 | Loss: 0.5048 | acc: 0.9375 
# step: 150/150 | Loss: 0.3413 | acc: 0.9375

### 必要に応じてモデルを保存

In [ ]:
torch.save({
        "model_state_dict": model.state_dict(),
        "config": config.__dict__,  # configも一緒に保存
        }, cls_model_filename)

モデルを保存していません


In [18]:
checkpoint = torch.load(cls_model_filename)
config = ModelConfig(tokenizer)
config.__dict__.update(checkpoint["config"])
model = DNN(config).to(device)
model.load_state_dict(checkpoint["model_state_dict"])

<All keys matched successfully>

### 検証

In [19]:
test_dataset = ClassificationDataset(data=df_test, tokenizer=tokenizer)
test_loader = DataLoader(
    dataset = test_dataset, 
    batch_size=len(test_dataset),
    shuffle = False,
    collate_fn=collate_wrapper,
    pin_memory=True
    )

In [20]:
model.eval()
test_batch = next(iter(test_loader))
input_ids = test_batch["ids"].to(device, non_blocking=True)
attention_mask = test_batch["attention_mask"].to(device, non_blocking=True)
label = test_batch["label"].to(device, non_blocking=True)
with torch.inference_mode():
    y = model(input_ids, attention_mask)
acc = accuracy(y, label)
print(f"検証精度：{acc}")

検証精度：0.9038461538461539
